# GAM发育曲线建模
对不同不利环境分组的脑指标进行GAM建模并可视化发育轨迹

In [ ]:
# 加载必要的库

library(mgcv)
library(ggplot2)
library(dplyr)
library(tidyr)
library(readr)
library(gratia)
cat("✓ 库已加载\n")

In [ ]:
# ==================== 用户配置 ====================
brain_file <- "/path/to/data/final/dmri_net_all_harmonized_siteqc_grade.csv"
covar_file <- "/path/to/data/raw/covs/dmri_combat_cov.csv"  # 协变量文件(包含年级)
brain_cols <- c(
    # "Visual1_FA",
    # "Visual2_FA",
    # "Auditory_FA",
    # "Somatomotor_FA",
    "Orbito-Affective_FA"
    # "Ventral-Multimodal_FA",
    # "Posterior-Multimodal_FA",
    # "Cingulo-Opercular_FA",
    # "Default_FA",
    # "Language_FA",
    # "Frontoparietal_FA",
    # "Dorsal-Attention_FA"
)  # 要分析的脑指标列名



#========================================================================================================================================
base_dir <- "/path/to/data/final/psm_trauma_results"  # PSM匹配结果目录
output_dir <- "/path/to/output/gam"  # 输出目录
groups <- c("不利", "无不利")
group_labels <- c("不利" = "Adverse", "无不利" = "Non-adverse")
plot_groups <- unname(group_labels[groups])

k_value <- 3  # GAM平滑项的基函数数量

dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)
cat(sprintf("输出目录: %s\n平滑项k值: %d\n分组: %s\n", output_dir, k_value, paste(groups, collapse=", ")))

In [ ]:
# # 加载数据
# brain_df <- read_csv(brain_file, show_col_types = FALSE)
# covar_df <- read_csv(covar_file, show_col_types = FALSE)

# # 标准化列名(去除空格)
# names(brain_df) <- trimws(names(brain_df))
# names(covar_df) <- trimws(names(covar_df))

# brain_subj_col <- names(brain_df)[1]
# covar_subj_col <- names(covar_df)[1]

# # 合并数据(假设协变量表中年级列名为"年级"或"grade")
# grade_col <- names(covar_df)[grep("年级|grade", names(covar_df), ignore.case = TRUE)][1]
# covar_subset <- covar_df[, c(covar_subj_col, grade_col)]
# names(covar_subset) <- c("subject_id", "grade")

# brain_df_renamed <- brain_df
# names(brain_df_renamed)[1] <- "subject_id"

# merged_df <- inner_join(brain_df_renamed, covar_subset, by = "subject_id")

# cat(sprintf("脑数据: %d 被试, %d 列\n", nrow(brain_df), ncol(brain_df)))
# cat(sprintf("协变量: %d 被试, 年级列: %s\n", nrow(covar_df), grade_col))
# cat(sprintf("合并后: %d 被试\n", nrow(merged_df)))

In [ ]:
# 加载数据
brain_df <- read_csv(brain_file, show_col_types = FALSE)
covar_df <- read_csv(covar_file, show_col_types = FALSE)

# 标准化列名(去除空格)
names(brain_df) <- trimws(names(brain_df))
names(covar_df) <- trimws(names(covar_df))

brain_subj_col <- names(brain_df)[1]
covar_subj_col <- names(covar_df)[1]

# 合并数据(查找年级或月龄列)
# 优先查找月龄，如无则使用年级
time_col <- "Age_months"
# time_col <- names(covar_df)[grep("月龄|age_month", names(covar_df), ignore.case = TRUE)][1]
# if (is.na(time_col)) {
#   time_col <- names(covar_df)[grep("年级|grade", names(covar_df), ignore.case = TRUE)][1]
# }

covar_subset <- covar_df[, c(covar_subj_col, time_col)]
names(covar_subset) <- c("subject_id", time_col)  # 保留原始时间变量名

brain_df_renamed <- brain_df
names(brain_df_renamed)[1] <- "subject_id"

merged_df <- inner_join(brain_df_renamed, covar_subset, by = "subject_id")

cat(sprintf("脑数据: %d 被试, %d 列\n", nrow(brain_df), ncol(brain_df)))
cat(sprintf("协变量: %d 被试, 时间变量: %s\n", nrow(covar_df), time_col))
cat(sprintf("合并后: %d 被试\n", nrow(merged_df)))

In [ ]:
# GAM建模和可视化函数
fit_gam_curves <- function(groups, brain_col) {
  # 读取各组被试并合并数据
  group_data_list <- lapply(groups, function(grp) {
    csv_path <- file.path(base_dir, paste0(grp, ".csv"))
    if (!file.exists(csv_path)) { cat(sprintf("  文件不存在: %s\n", csv_path)); return(NULL) }
    
    grp_df <- read_csv(csv_path, show_col_types = FALSE)
    grp_subjects <- as.character(grp_df[[1]])
    
    grp_data <- merged_df %>%
      filter(subject_id %in% grp_subjects) %>%
      select(subject_id, all_of(time_col), all_of(brain_col)) %>%
      drop_na() %>%
      mutate(group = unname(group_labels[grp]))
    
    if (nrow(grp_data) > 0) return(grp_data) else return(NULL)
  })
  
  # 合并所有组数据
  all_data <- bind_rows(group_data_list[!sapply(group_data_list, is.null)])
  if (nrow(all_data) < 10) return(NULL)
  
  all_data$group <- factor(all_data$group, levels = plot_groups)
  
  # 重命名脑指标列为安全名称
  safe_col <- "brain_metric"
  names(all_data)[names(all_data) == brain_col] <- safe_col
  
  # ========== 检验方法 ==========
  # 使用动态公式，time_col为实际时间变量名（如"月龄"）
  # 1. 轨迹形状差异检验
  gam_shape <- gam(as.formula(sprintf("%s ~ group + s(`%s`, by=group, k=%d)", safe_col, time_col, k_value)),
                   data=all_data, method="REML")
  gam_base <- gam(as.formula(sprintf("%s ~ group + s(`%s`, k=%d)", safe_col, time_col, k_value)),
                  data=all_data, method="REML")
  anova_shape <- anova(gam_base, gam_shape, test="F")
  
  
  # 2. 组间截距差异检验
  gam_null <- gam(as.formula(sprintf("%s ~ s(`%s`, k=%d)", safe_col, time_col, k_value)),
                  data=all_data, method="REML")
  anova_intercept <- anova(gam_null, gam_base, test="F")
  
  # 检验结果
  test_result <- list(
    shape_F = anova_shape$F[2],
    shape_df1 = anova_shape$Df[2],
    shape_df2 = anova_shape$`Resid. Df`[2],
    shape_p = anova_shape$`Pr(>F)`[2],
    intercept_F = anova_intercept$F[2],
    intercept_df1 = anova_intercept$Df[2],
    intercept_df2 = anova_intercept$`Resid. Df`[2],
    intercept_p = anova_intercept$`Pr(>F)`[2]
  )
  
  # 用形状差异模型作图
  time_range <- seq(min(all_data[[time_col]]), max(all_data[[time_col]]), length.out = 100)
  pred_data <- expand.grid(time_val = time_range, group = factor(plot_groups, levels = plot_groups))
  names(pred_data)[1] <- time_col  # 使用实际时间变量名
  pred_data$fitted <- predict(gam_shape, newdata = pred_data)
  
  # 模型摘要
  model_summary <- summary(gam_shape)
  
  # 整理参数项结果
  results_df <- data.frame(
    term = rownames(model_summary$p.table),
    estimate = model_summary$p.table[, "Estimate"],
    se = model_summary$p.table[, "Std. Error"],
    t_value = model_summary$p.table[, "t value"],
    p_value = model_summary$p.table[, "Pr(>|t|)"]
  )
  
  # 整理平滑项结果
  smooth_results <- data.frame(
    smooth_term = rownames(model_summary$s.table),
    edf = model_summary$s.table[, "edf"],
    ref_df = model_summary$s.table[, "Ref.df"],
    F_value = model_summary$s.table[, "F"],
    p_value = model_summary$s.table[, "p-value"]
  )
  
  model_stats <- data.frame(
    R_squared = model_summary$r.sq,
    deviance_explained = model_summary$dev.expl,
    n_obs = nrow(all_data),
    AIC_shape = AIC(gam_shape),
    AIC_base = AIC(gam_base),
    AIC_null = AIC(gam_null)
  )
  
  list(all_data = all_data, pred_data = pred_data, 
       results = results_df, smooth = smooth_results, stats = model_stats,
       test_result = test_result,
       original_col = brain_col, safe_col = safe_col)
}

cat("✓ 函数定义完成\n")

In [ ]:
# # 绘制三组被试在各年级的人数分布
# grade_dist <- lapply(groups, function(grp) {
#   grp_df <- read_csv(file.path(base_dir, paste0(grp, ".csv")), show_col_types=FALSE)
#   merged_df %>% filter(subject_id %in% grp_df[[1]]) %>%
#     select(subject_id, grade) %>% distinct() %>% mutate(group=grp)
# }) %>% bind_rows() %>% count(group, grade) %>%
#   mutate(group = factor(group, levels=groups))

# color_map <- c("高度不利"="#E74C3C", "轻度不利"="#F39C12", "无不利"="#27AE60")

# ggplot(grade_dist, aes(x=factor(grade), y=n, fill=group)) +
#   geom_bar(stat="identity", position="dodge", width=0.7) +
#   geom_text(aes(label=n), position=position_dodge(0.7), vjust=-0.5, size=3.5) +
#   scale_fill_manual(values=color_map) +
#   labs(title="三组被试在各年级的人数分布", x="年级", y="人数", fill="组别") +
#   theme_minimal() +
#   theme(plot.title=element_text(hjust=0.5, face="bold"),
#         panel.background=element_rect(fill="white", color=NA),
#         plot.background=element_rect(fill="white", color=NA))

# ggsave(file.path(output_dir, "三组年级分布.png"), width=10, height=6, dpi=300, bg="white")
# cat("已保存: 三组年级分布.png\n")

In [ ]:
# 主分析循环
all_gam_tests <- data.frame()

for (brain_col in brain_cols) {
  cat(sprintf(
    "\n%s\n脑指标: %s\n%s\n",
    paste(rep("=", 60), collapse = ""),
    brain_col,
    paste(rep("=", 60), collapse = "")
  ))

  # 拟合GAM模型
  result <- tryCatch(
    fit_gam_curves(groups, brain_col),
    error = function(e) {
      cat(sprintf("  错误: %s\n", e$message))
      NULL
    }
  )

  if (is.null(result)) {
    cat("  数据不足或拟合失败, 跳过\n")
    next
  }

  # 输出检验结果
  tt <- result$test_result
  cat(sprintf(
    "\n  【轨迹形状差异检验】 y ~ group + s(%s, by=group) vs y ~ group + s(%s)\n",
    time_col,
    time_col
  ))
  cat(sprintf(
    "  F(%.3f, %.3f)=%.3f, p=%.4f %s\n",
    tt$shape_df1,
    tt$shape_df2,
    tt$shape_F,
    tt$shape_p,
    ifelse(tt$shape_p < 0.05, "*", "")
  ))
  cat(sprintf(
    "  【组间截距差异检验】 y ~ group + s(%s) vs y ~ s(%s)\n",
    time_col,
    time_col
  ))
  cat(sprintf(
    "  F(%.3f, %.3f)=%.3f, p=%.4f %s\n",
    tt$intercept_df1,
    tt$intercept_df2,
    tt$intercept_F,
    tt$intercept_p,
    ifelse(tt$intercept_p < 0.05, "*", "")
  ))

  # 汇总检验结果
  all_gam_tests <- rbind(
    all_gam_tests,
    data.frame(
      脑指标 = brain_col,
      轨迹形状差异_F = round(tt$shape_F, 3),
      轨迹形状差异_df1 = tt$shape_df1,
      轨迹形状差异_df2 = tt$shape_df2,
      轨迹形状差异_p = round(tt$shape_p, 4),
      形状显著 = ifelse(tt$shape_p < 0.05, "是", "否"),
      截距差异_F = round(tt$intercept_F, 3),
      截距差异_df1 = tt$intercept_df1,
      截距差异_df2 = tt$intercept_df2,
      截距差异_p = round(tt$intercept_p, 4),
      截距显著 = ifelse(tt$intercept_p < 0.05, "是", "否")
    )
  )

  # 保存详细结果
  result_file <- file.path(output_dir, sprintf("%s_结果.csv", brain_col))
  write_csv(
    bind_rows(
      mutate(result$results, type = "parametric"),
      mutate(result$smooth, type = "smooth") %>%
        rename(term = smooth_term, estimate = edf, se = ref_df, t_value = F_value)
    ),
    result_file
  )

  # 保存模型统计
  stats_file <- file.path(output_dir, sprintf("%s_模型统计.csv", brain_col))
  write_csv(result$stats, stats_file)

  cat(sprintf("  样本量: %d, R² = %.3f\n", result$stats$n_obs, result$stats$R_squared))

  # 计算每组样本量并创建带n的标签
  group_counts <- result$all_data %>%
    group_by(group) %>%
    summarise(n = n(), .groups = "drop")

  # 创建颜色映射
  color_map <- c(
    "Adverse" = "#C44E52",
    "Non-adverse" = "#4C72B0"
  )

  # 创建带样本量的标签映射
  label_map <- setNames(
    sprintf("%s (n=%d)", group_counts$group, group_counts$n),
    group_counts$group
  )

  # 添加group_label列
  result$all_data <- result$all_data %>%
    mutate(group_label = label_map[as.character(group)])

  result$pred_data <- result$pred_data %>%
    left_join(group_counts, by = "group") %>%
    mutate(group_label = sprintf("%s (n=%d)", group, n))

  # 设置图例标签的颜色
  label_colors <- setNames(
    color_map[group_counts$group],
    label_map[group_counts$group]
  )

  # 构建显著性标注
  sig_label <- sprintf(
    "Curve shape difference: p=%.3f%s\nGroup intercept difference: p=%.3f%s",
    tt$shape_p,
    ifelse(tt$shape_p < 0.05, "*", ""),
    tt$intercept_p,
    ifelse(tt$intercept_p < 0.05, "*", "")
  )

  # 绘制发育曲线（白色背景）
  # 绘制发育曲线1：散点+拟合曲线
  p1 <- ggplot() +
    geom_point(
      data = result$all_data,
      aes(
        x = .data[[time_col]],
        y = .data[[result$safe_col]],
        color = group_label
      ),
      alpha = 0.4,
      size = 1.5
    ) +
    geom_line(
      data = result$pred_data,
      aes(x = .data[[time_col]], y = fitted, color = group_label),
      linewidth = 1.2
    ) +
    scale_color_manual(values = label_colors, breaks = label_map[plot_groups]) +
    labs(
      title = sprintf("Group-specific curves: %s", brain_col),
      subtitle = sig_label,
      x = "Age (months)",
      y = brain_col,
      color = "Group"
    ) +
    theme_minimal(base_size = 16) +
    theme(
      legend.position = "right",
      legend.title = element_text(face = "bold", size = 16),
      legend.text = element_text(size = 15),
      plot.title = element_text(hjust = 0.5, face = "bold", size = 22),
      plot.subtitle = element_text(hjust = 0.5, size = 16, color = "gray30"),
      axis.title = element_text(size = 18),
      axis.text = element_text(size = 15),
      panel.background = element_rect(fill = "white", color = NA),
      plot.background = element_rect(fill = "white", color = NA)
    )

  plot_file1 <- file.path(output_dir, sprintf("%s_发育曲线.png", brain_col))
  ggsave(plot_file1, p1, width = 12, height = 6, dpi = 300, bg = "white")
  cat(sprintf("  已保存: %s\n", basename(plot_file1)))

  # 绘制发育曲线2：拟合曲线+置信带（无散点）
  # 重新拟合模型以获取置信区间
  gam_shape <- gam(
    as.formula(sprintf("brain_metric ~ group + s(`%s`, by=group, k=%d)", time_col, k_value)),
    data = result$all_data,
    method = "REML"
  )
  pred_ci <- predict(gam_shape, newdata = result$pred_data, se.fit = TRUE)
  result$pred_data$fitted <- as.numeric(pred_ci$fit)
  result$pred_data$se <- as.numeric(pred_ci$se.fit)
  result$pred_data$lower <- result$pred_data$fitted - 1.96 * result$pred_data$se
  result$pred_data$upper <- result$pred_data$fitted + 1.96 * result$pred_data$se


  # 保存每组拟合曲线及其逐点95%置信区间
  group_curve_table <- result$pred_data %>%
    transmute(
      brain_metric = brain_col,
      group = as.character(group),
      age_months = .data[[time_col]],
      age_years = .data[[time_col]] / 12,
      fitted_value = fitted,
      standard_error = se,
      lower_95_ci = lower,
      upper_95_ci = upper,
      ci_level = 0.95,
      group_n = .data[["n"]]
    )
  group_curve_file <- file.path(
    output_dir,
    sprintf("%s_group_curve_pointwise_ci.csv", brain_col)
  )
  write_csv(group_curve_table, group_curve_file)
  cat(sprintf("  Saved: %s\n", basename(group_curve_file)))

  # 颜色填充映射
  fill_colors <- label_colors

  p2 <- ggplot(
    result$pred_data,
    aes(x = .data[[time_col]], y = fitted, color = group_label, fill = group_label)
  ) +
    geom_ribbon(aes(ymin = lower, ymax = upper), alpha = 0.2, color = NA) +
    geom_line(linewidth = 1.2) +
    scale_color_manual(values = label_colors, breaks = label_map[plot_groups]) +
    scale_fill_manual(values = fill_colors, breaks = label_map[plot_groups]) +
    labs(
      title = sprintf("Group-specific curves: %s", brain_col),
      subtitle = sig_label,
      x = "Age (months)",
      y = brain_col,
      color = "Group",
      fill = "Group"
    ) +
    theme_minimal(base_size = 16) +
    theme(
      legend.position = "right",
      legend.title = element_text(face = "bold", size = 16),
      legend.text = element_text(size = 15),
      plot.title = element_text(hjust = 0.5, face = "bold", size = 22),
      plot.subtitle = element_text(hjust = 0.5, size = 16, color = "gray30"),
      axis.title = element_text(size = 18),
      axis.text = element_text(size = 15),
      panel.background = element_rect(fill = "white", color = NA),
      plot.background = element_rect(fill = "white", color = NA)
    )

  plot_file2 <- file.path(output_dir, sprintf("%s_发育曲线_置信带.png", brain_col))
  ggsave(plot_file2, p2, width = 12, height = 6, dpi = 300, bg = "white")
  cat(sprintf("  已保存: %s\n", basename(plot_file2)))
}

# 保存所有检验汇总
if (nrow(all_gam_tests) > 0) {
  write_csv(all_gam_tests, file.path(output_dir, "GAM轨迹检验汇总.csv"))
  cat("\n=== GAM轨迹检验汇总 ===\n")
  print(all_gam_tests)
}

cat("\n✓ 所有分析完成!\n")

In [ ]:
# 事后检验：使用gratia绘制组间平滑差异曲线（对所有脑指标）
# install.packages(c("gratia", "patchwork"))
library(gratia)
library(patchwork)

for (brain_col in brain_cols) {
  cat(sprintf("\n[事后检验] %s\n", brain_col))
  res <- fit_gam_curves(groups, brain_col)
  if (is.null(res)) next
  
  # 拟合模型（使用实际时间变量名）
  gam_mod <- gam(as.formula(sprintf("brain_metric ~ group + s(`%s`, by=group, k=%d)", time_col, k_value)), 
                 data=res$all_data, method="REML")
  
   # 计算组间平滑差值及逐点95%置信区间（不包含组间参数截距差异）
  smooth_name <- sprintf("s(%s)", time_col)
  diff_method_args <- names(formals(getS3method("difference_smooths", "gam")))
  diff_call_args <- list(
    model = gam_mod,
    n = 100,
    ci_level = 0.95
  )
  if ("select" %in% diff_method_args) {
    diff_call_args$select <- smooth_name
  } else {
    diff_call_args$smooth <- smooth_name
  }
  if ("group_means" %in% diff_method_args) {
    diff_call_args$group_means <- FALSE
  }
  diff_result <- do.call(difference_smooths, diff_call_args)
  diff_raw <- as.data.frame(diff_result)

  # gratia 0.9.0起为结果字段增加了“.”前缀；统一新旧版本字段名
  old_diff_names <- c(
    "smooth", "by", "level_1", "level_2",
    "diff", "se", "lower", "upper"
  )
  new_diff_names <- c(
    ".smooth", ".by", ".level_1", ".level_2",
    ".diff", ".se", ".lower_ci", ".upper_ci"
  )
  for (i in seq_along(old_diff_names)) {
    old_name <- old_diff_names[i]
    new_name <- new_diff_names[i]
    if (old_name %in% names(diff_raw) && !new_name %in% names(diff_raw)) {
      names(diff_raw)[names(diff_raw) == old_name] <- new_name
    }
  }


  required_diff_cols <- c(
    ".smooth", ".by", ".level_1", ".level_2",
    ".diff", ".se", ".lower_ci", ".upper_ci", time_col
  )
  missing_diff_cols <- setdiff(required_diff_cols, names(diff_raw))
  if (length(missing_diff_cols) > 0) {
    stop(sprintf(
      "difference_smooths() output is missing required columns: %s",
      paste(missing_diff_cols, collapse = ", ")
    ))
  }

  difference_table <- diff_raw %>%
    transmute(
      brain_metric = brain_col,
      smooth_term = .data[[".smooth"]],
      by_variable = .data[[".by"]],
      group_1 = .data[[".level_1"]],
      group_2 = .data[[".level_2"]],
      contrast = paste(.data[[".level_1"]], .data[[".level_2"]], sep = " - "),
      age_months = .data[[time_col]],
      age_years = .data[[time_col]] / 12,
      smooth_difference = .data[[".diff"]],
      standard_error = .data[[".se"]],
      lower_95_ci = .data[[".lower_ci"]],
      upper_95_ci = .data[[".upper_ci"]],
      ci_level = 0.95,
      group_means_included = FALSE
    ) %>%
    mutate(
      significant = !is.na(lower_95_ci) & !is.na(upper_95_ci) &
        (lower_95_ci > 0 | upper_95_ci < 0),
      direction = case_when(
        lower_95_ci > 0 ~ paste(group_1, ">", group_2),
        upper_95_ci < 0 ~ paste(group_1, "<", group_2),
        TRUE ~ "Not significant"
      )
    ) %>%
    arrange(group_1, group_2, age_months)

  difference_file <- file.path(
    output_dir,
    sprintf("%s_smooth_difference_pointwise_ci.csv", brain_col)
  )
  write_csv(difference_table, difference_file)

  # 将连续的显著网格点汇总为显著年龄窗口
  window_table <- difference_table %>%
    group_by(brain_metric, smooth_term, by_variable, group_1, group_2, contrast) %>%
    arrange(age_months, .by_group = TRUE) %>%
    mutate(
      window_break = significant != lag(significant, default = first(significant)) |
        direction != lag(direction, default = first(direction)),
      window_id = cumsum(window_break) + 1L
    ) %>%
    filter(significant) %>%
    group_by(
      brain_metric, smooth_term, by_variable, group_1, group_2, contrast,
      window_id, direction
    ) %>%
    summarise(
      start_age_months = min(age_months),
      end_age_months = max(age_months),
      start_age_years = min(age_years),
      end_age_years = max(age_years),
      window_duration_months = max(age_months) - min(age_months),
      n_grid_points = n(),
      age_grid_step_months = ifelse(n() > 1, median(diff(age_months)), NA_real_),
      minimum_difference = min(smooth_difference),
      maximum_difference = max(smooth_difference),
      ci_level = first(ci_level),
      group_means_included = first(group_means_included),
      .groups = "drop"
    ) %>%
    select(-window_id)

  window_file <- file.path(
    output_dir,
    sprintf("%s_smooth_difference_significant_age_windows.csv", brain_col)
  )
  write_csv(window_table, window_file)

  cat(sprintf("  Saved: %s\n", basename(difference_file)))
  cat(sprintf("  Saved: %s\n", basename(window_file)))

  # 绘制差异曲线，添加y=0参考线
  p <- draw(diff_result) &
    geom_hline(yintercept = 0, linetype = "dashed", color = "red", linewidth = 0.5)

  p <- p + plot_annotation(
    title = sprintf("%s: Between-group smooth difference", brain_col),
    theme = theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 22))
  ) &
    theme(
      panel.background = element_rect(fill = "white", color = NA),
      plot.background = element_rect(fill = "white", color = NA),
      axis.line = element_line(color = "black", linewidth = 0.5),
      axis.text = element_text(color = "black", size = 15),
      axis.title = element_text(color = "black", size = 18),
      strip.text = element_text(size = 16, face = "bold"),
      axis.ticks = element_line(color = "black")
    )

  ggsave(
    file.path(output_dir, sprintf("%s_差异曲线.png", brain_col)),
    p,
    width = 14,
    height = 5,
    dpi = 300,
    bg = "white"
  )
  cat(sprintf("  已保存: %s_差异曲线.png\n", brain_col))
}
cat("\n✓ 事后检验完成!\n")